In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import expr , col ,countDistinct,count

spark = SparkSession.builder\
                    .appName("Read Customer")\
                    .getOrCreate()

class CustomerAnalysis:
    def __init__(self):
        pass

    def readCustomer(self):
        return (
            spark.read.format("csv")
                      .option("header" , "true")
                      .load(path = '/Volumes/dev/spark_db/datasets/spark_programming/data/customers.csv')
        )
    def schemaFix(self,customer_df):
        return (
            customer_df.withColumns (
                {
                'registration_date' : expr("to_date(registration_date , 'yyyy-MM-dd')"),
                'is_active' : expr("cast(is_active as boolean)")   # col("is_active").cast("boolean")
                 }
            )
                      
        )
    def nullFix(self,df):
        return (
            df.withColumns(
                {
                    'city' : expr("nvl(city , 'Unknown')"),
                    'state' : expr("nvl(state , 'Unknown')"),
                    'country' : expr("nvl(country , 'Unknown')")
                }
            )
        )               # df.fillna({'city':'Unknown' , 'state':'Unknown' , 'country' : 'Unknown'})

    def registationSplit(self, null_fix_df):
        return (
            null_fix_df.withColumns(
                {
                    "registration_year" : expr("year(registration_date)"),
                    "registration_month" : expr("month(registration_date)")
                }
            )
        )

    def uniqueCityStateCountry(self , reg_df):
        return (
            reg_df.select(countDistinct("city")).collect()[0][0],
            reg_df.select(countDistinct("state")).collect()[0][0],
            reg_df.select(countDistinct("country")).collect()[0][0]
        )

    def aggregate(self , reg_df , para : str):
        if para == 'city':
            return (
                reg_df.groupBy("city").count().alias("count")                #.agg(count("customer_id").alias("count"))
                    .orderBy(col("count").desc())
            )
        elif para == 'state':
            return (
                reg_df.groupBy("city").count().alias("count")                
                    .orderBy(col("count").desc())
            )
        elif para == 'country':
            return (
                reg_df.groupBy("state","country").count().alias("count")                
                    .orderBy(col("count").desc())
            )

    def shemaInfo(self , customer_df):
        customer_df.show(5)
        customer_df.printSchema()

    def main(self):
        customer_df = self.readCustomer()
        schema_fixed_df = self.schemaFix(customer_df)
        nullFixed = self.nullFix(schema_fixed_df)
        reg_df = self.registationSplit(nullFixed)
        agg = self.aggregate(reg_df,'country')
        self.shemaInfo(agg)

        resultes = self.uniqueCityStateCountry(reg_df)
        print(resultes)

a = CustomerAnalysis()
a.main()
